# 01 · Build the baseline-anchored prodromal SAA dataset

Reads the PPMI curated data cut and the raw SAA result files, and writes the analysis files used by notebooks 02 and 03.

**Inputs** (`<BASE_DIR>/data/`): `PPMI_Curated_Data_Cut_Public_20260511.xlsx`, `SAA_Biospecimen_Analysis_Results_29Jul2026.csv`, `Kinetic_data_for_SAA_Results_29Jul2026.csv`

**Outputs** (`<BASE_DIR>/results_baseline/`): `prodromal_saa_baseline_core.csv` (one row per participant), `prodromal_saa_longitudinal.csv` (one row per visit), `prodromal_saa_baseline_all_curated.csv`, `analytic_group_counts_baseline.csv`, `visit_availability.csv`, `positivity_by_enrollment_year.csv`, `metadata_baseline.json`

In [ ]:
# Works in Google Colab (mounts Drive) and locally. Set the project folder here,
# or through the PPMI_SAA_BASE environment variable.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

BASE_DIR = os.environ.get('PPMI_SAA_BASE',
                          'PPMI_Projects/prodromal_SAA_groups')

In [ ]:
# PPMI prodromal SAA dataset builder -- baseline-anchored
#
#   1. SAA status comes from the curated CSFSAA field on the BL row.
#      CSFSAA is already the assay-combined result ("prioritizing the newest
#      assay, 24hr or 35hr"), so no 237/299 precedence logic is needed.
#   2. Positive is TYPE-AGNOSTIC: codes 1 (LBD-like) and 3 (MSA-like) both
#      map to Positive. Code 0 -> Negative. Codes 2 (Inconclusive) and
#      '.'/blank (Unknown) are excluded from the analytic sample.
#      The raw code is retained in CSFSAA_code so type is recoverable.
#   3. Raw kinetics are restricted to the baseline window, so every variable
#      in a row refers to the same visit. No latest-visit selection anywhere.
#   4. LONGITUDINAL: follow-up visits are retained. Two files are produced --
#      a wide file (one row per participant, baseline only) and a long file
#      (one row per participant-visit). Baseline SAA status is carried on
#      every follow-up row as a fixed grouping variable (*_BL suffix), so it
#      can be used as the exposure in a repeated-measures or survival model.

from __future__ import annotations
from zipfile import ZipFile
from xml.etree import ElementTree as ET
from pathlib import Path
from statistics import mean, median
from collections import Counter
import csv, re, json

BASE = Path(BASE_DIR)

CURATED = BASE / "data/PPMI_Curated_Data_Cut_Public_20260511.xlsx"
BIO     = BASE / "data/SAA_Biospecimen_Analysis_Results_29Jul2026.csv"
KIN     = BASE / "data/Kinetic_data_for_SAA_Results_29Jul2026.csv"
OUT     = BASE / "results_baseline"
OUT.mkdir(parents=True, exist_ok=True)

for _p in (CURATED, BIO, KIN):
    if not _p.exists():
        raise FileNotFoundError(f"Missing input: {_p}")

REPLICATE_AGGREGATION = 'mean'

# Baseline is BL only -- screening (SC) is deliberately excluded. Note the
# cost: Project 299 records most of its baseline draws under SC, so excluding
# SC removes almost all 35-hour kinetics and baseline kinetics become mostly
# Project 237 (24-hour assay); see with_baseline_kinetics_* in the metadata.
# Status is NOT affected -- it comes from the curated CSFSAA field on the BL
# row, independent of this flag. Set to {'SC', 'BL'} to include screening.
BASELINE_EVENTS = {'BL'}

# Curated follow-up visits, in order. YEAR maps 1:1 onto these (BL=0 ... V22=15).
VISIT_ORDER = {'BL': 0, 'V04': 1, 'V06': 2, 'V08': 3, 'V10': 4, 'V12': 5, 'V13': 6, 'V14': 7,
               'V15': 8, 'V16': 9, 'V17': 10, 'V18': 11, 'V19': 12, 'V20': 13, 'V21': 14, 'V22': 15}

POSITIVE_CODES = {'1', '3'}     # 1 = Positive LBD-like, 3 = Positive MSA-like
NEGATIVE_CODES = {'0'}          # 0 = Negative
CODE_LABELS = {'0': 'Negative', '1': 'Positive LBD-like', '3': 'Positive MSA-like'}

GROUP_LABELS = {
    'A1': 'PSG-confirmed RBD with hyposmia',
    'A2': 'PSG-confirmed RBD without hyposmia',
    'B':  'Hyposmia subgroup with possible RBD',
    'C':  'Hyposmia subgroup without self-endorsed RBD',
    '':   'Missing / unclassified analytic subgroup',
}

CORE_CURATED_COLUMNS = [
    'SITE','PATNO','COHORT','subgroup','enroll_phase','enroll_source','analytic_subgroup','HIQ_RBD',
    'study_status','PRIMDIAG','EVENT_ID','YEAR','visit_date',
    'age_at_visit','SEX','EDUCYRS','race','fampd_bin',
    'BMI',
    'upsit','upsit_pctl','upsit_pctl15','moca','ess','rem','gds',
    'stai','scopa','orthostasis','hy','pigd','td_pigd','updrs1_score','updrs2_score','updrs3_score',
    'updrs4_score','updrs_totscore',
    'CSFSAA','CSFSAA_assay','APOE_e4'
]

NS = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'


def colidx(ref: str) -> int:
    m = re.match(r'([A-Z]+)', ref); n = 0
    for ch in m.group(1): n = n * 26 + ord(ch) - 64
    return n - 1


def shared_strings(z: ZipFile):
    if 'xl/sharedStrings.xml' not in z.namelist(): return []
    root = ET.fromstring(z.read('xl/sharedStrings.xml'))
    return [''.join(t.text or '' for t in si.iter(NS + 't')) for si in root.findall(NS + 'si')]


def xlsx_sheet_rows(path: Path, sheet_xml: str):
    with ZipFile(path) as z:
        shared = shared_strings(z)
        with z.open(sheet_xml) as f:
            for _, elem in ET.iterparse(f, events=('end',)):
                if elem.tag != NS + 'row': continue
                d = {}
                for c in elem.findall(NS + 'c'):
                    i = colidx(c.attrib['r']); typ = c.attrib.get('t'); v = c.find(NS + 'v')
                    if typ == 's' and v is not None: val = shared[int(v.text)]
                    elif typ == 'inlineStr': val = ''.join(t.text or '' for t in c.iter(NS + 't'))
                    elif v is not None: val = v.text
                    else: val = ''
                    d[i] = val
                yield d
                elem.clear()


def normalize_patno(v):
    s = str(v).strip()
    return s[:-2] if s.endswith('.0') else s


def num(v):
    try:
        s = str(v).strip()
        return float(s) if s != '' else None
    except Exception:
        return None


def visit_year(v):
    """Year out of a curated visit_date like '03/2023'."""
    for part in re.split(r'[-/]', str(v)):
        if len(part) == 4 and part.isdigit():
            return part
    return ''


def agg(vals):
    x = [num(v) for v in vals]
    x = [v for v in x if v is not None]
    if not x: return {'mean': '', 'median': '', 'n': 0}
    return {'mean': mean(x), 'median': median(x), 'n': len(x)}


def read_phase1_2():
    """Phase-1 and Phase-2 prodromal participants (COHORT 4): every visit, plus a BL lookup."""
    it = xlsx_sheet_rows(CURATED, 'xl/worksheets/sheet1.xml')
    hd = next(it)
    headers = [hd.get(i, '') for i in range(max(hd) + 1)]
    idx = {h: i for i, h in enumerate(headers)}
    visits, baseline = [], {}
    for d in it:
        if d.get(idx['COHORT'], '') != '4' or d.get(idx['enroll_phase'], '') not in ('1', '2'):
            continue
        r = {h: d.get(i, '') for h, i in idx.items()}
        r['PATNO'] = normalize_patno(r['PATNO'])
        visits.append(r)
        if r['EVENT_ID'] == 'BL':
            baseline[r['PATNO']] = r
    return headers, visits, baseline


def kinetics237(ids):
    """Project 237 replicate kinetics from the baseline visit only."""
    with BIO.open(encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    metric_cols = {
        'Fmax':           ['Fmax_24h_Rep1', 'Fmax_24h_Rep2', 'Fmax_24h_Rep3'],
        'TTT':            ['TTT_24h_Rep1', 'TTT_24h_Rep2', 'TTT_24h_Rep3'],
        'AUC':            ['AUC_24h_Rep1', 'AUC_24h_Rep2', 'AUC_24h_Rep3'],
        'TimeToMaxSlope': ['TSmax_24h_Rep1', 'TSmax_24h_Rep2', 'TSmax_24h_Rep3'],
        'MaxSlope':       ['SLOPEMax_24h_Rep1', 'SLOPEMax_24h_Rep2', 'SLOPEMax_24h_Rep3'],
    }
    best = {}
    for r in rows:
        p = normalize_patno(r['PATNO'])
        if (p not in ids or r['COHORT'] != 'Prodromal' or r['PROJECTID'] != '237'
                or r['CLINICAL_EVENT'] not in BASELINE_EVENTS):
            continue
        # if the baseline sample was re-run, keep the most recent run
        if p not in best or r['RUNDATE'] > best[p]['RUNDATE']:
            best[p] = r
    out = {}
    for p, r in best.items():
        rec = {'PATNO': p, 'P237_SAA_Status': r['SAA_Status'], 'P237_SAA_Type': r.get('SAA_Type', ''),
               'P237_Event': r['CLINICAL_EVENT'], 'P237_RunDate': r['RUNDATE'],
               'P237_Method': r.get('SAAMethod', '')}
        ns = []
        for m, cols in metric_cols.items():
            a = agg([r.get(c, '') for c in cols]); ns.append(a['n'])
            rec[f'P237_{m}_mean'] = a['mean']; rec[f'P237_{m}_median'] = a['median']; rec[f'P237_{m}_n'] = a['n']
        rec['P237_Replicates_MaxN'] = max(ns) if ns else 0
        out[p] = rec
    return out


def kinetics299(ids):
    """Project 299 well-level kinetics from the baseline visit only."""
    with KIN.open(encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    metric_cols = {'Fmax': 'Fmax', 'TTT': 'TTT', 'AUC': 'AUC_Fluoro',
                   'TimeToMaxSlope': 'Time_To_Max_Slope', 'MaxSlope': 'Max_Slope'}
    by = {}
    for r in rows:
        p = normalize_patno(r['PATNO'])
        if (p not in ids or r['COHORT'] != 'Prodromal' or r['PROJECTID'] != '299'
                or r['CLINICAL_EVENT'] not in BASELINE_EVENTS):
            continue
        by.setdefault(p, []).append(r)
    out = {}
    for p, prs in by.items():
        md = max(r['RUNDATE'] for r in prs)
        chosen = [r for r in prs if r['RUNDATE'] == md]
        rec = {'PATNO': p,
               'P299_SAA_Result_Types': '|'.join(sorted({r['SAA_Result'] for r in chosen if r['SAA_Result']})),
               'P299_Event': chosen[0]['CLINICAL_EVENT'], 'P299_RunDate': md,
               'P299_Method': chosen[0].get('SAAMethod', ''), 'P299_Replicate_Rows': len(chosen)}
        for m, c in metric_cols.items():
            a = agg([r.get(c, '') for r in chosen])
            rec[f'P299_{m}_mean'] = a['mean']; rec[f'P299_{m}_median'] = a['median']; rec[f'P299_{m}_n'] = a['n']
        out[p] = rec
    return out


_METRIC_PATTERNS = [
    ('TimeToMaxSlope', [r'tsmax', r'time.?to.?max.?slope']),
    ('MaxSlope',       [r'slope.?max', r'max.?slope']),
    ('Fmax',           [r'f.?max']),
    ('TTT',            [r'ttt', r'time.?to.?threshold']),
    ('AUC',            [r'auc']),
]


def _metric_of(col):
    """Map a raw column name onto a kinetic metric, or None. Order matters:
    time-to-max-slope is tested before max-slope."""
    c = col.strip().lower()
    for metric, pats in _METRIC_PATTERNS:
        if any(re.search(p, c) for p in pats):
            return metric
    return None


def kinetics155(ids):
    """Project 155 (150-hour pilot) kinetics from the baseline visit only.
    The pilot predates the 237/299 column conventions, so its kinetic columns
    are detected by pattern and both raw files are scanned. Aggregation mirrors
    299: keep the latest run date, then average over every replicate row and
    replicate column of that run."""
    out = {}
    for path in (BIO, KIN):
        with path.open(encoding='utf-8-sig', newline='') as f:
            rows = list(csv.DictReader(f))
        sel = []
        for r in rows:
            p = normalize_patno(r.get('PATNO', ''))
            coh = (r.get('COHORT', '') or '').strip().lower()
            if (p not in ids or str(r.get('PROJECTID', '')).strip() != '155'
                    or (coh and coh != 'prodromal')
                    or (r.get('CLINICAL_EVENT', '') or '') not in BASELINE_EVENTS):
                continue
            sel.append(r)
        if not sel:
            continue
        colmap = {}
        for c in sel[0].keys():
            m = _metric_of(c)
            if m and any(num(r.get(c, '')) is not None for r in sel):
                colmap.setdefault(m, []).append(c)
        by = {}
        for r in sel:
            by.setdefault(normalize_patno(r['PATNO']), []).append(r)
        for p, prs in by.items():
            if p in out:
                continue
            md = max((r.get('RUNDATE', '') or '') for r in prs)
            chosen = [r for r in prs if (r.get('RUNDATE', '') or '') == md]
            rec = {'PATNO': p,
                   'P155_SAA_Status': '|'.join(sorted({(r.get('SAA_Status') or r.get('SAA_Result') or '').strip()
                                                       for r in chosen} - {''})),
                   'P155_SAA_Type': '|'.join(sorted({(r.get('SAA_Type', '') or '').strip()
                                                     for r in chosen} - {''})),
                   'P155_Event': chosen[0].get('CLINICAL_EVENT', ''), 'P155_RunDate': md,
                   'P155_Method': chosen[0].get('SAAMethod', ''), 'P155_Replicate_Rows': len(chosen)}
            for m in ['Fmax', 'TTT', 'AUC', 'TimeToMaxSlope', 'MaxSlope']:
                a = agg([r.get(c, '') for r in chosen for c in colmap.get(m, [])])
                rec[f'P155_{m}_mean'] = a['mean']; rec[f'P155_{m}_median'] = a['median']; rec[f'P155_{m}_n'] = a['n']
            out[p] = rec
    return out


def build():
    headers, visits, baseline = read_phase1_2()
    all_ids = set(baseline)

    # --- status from the BL row, type-agnostic ---
    final_ids, excluded = [], Counter()
    for p in all_ids:
        code = baseline[p].get('CSFSAA', '')
        if code in POSITIVE_CODES or code in NEGATIVE_CODES:
            final_ids.append(p)
        else:
            excluded[code or '<blank>'] += 1
    final_ids.sort(key=lambda x: int(x) if x.isdigit() else x)

    k237 = kinetics237(set(final_ids))
    k299 = kinetics299(set(final_ids))
    k155 = kinetics155(set(final_ids))

    saa_cols = (['SAA_Status', 'SAA_Status_Code', 'CSFSAA_code', 'CSFSAA_code_label', 'BL_enroll_year',
                 'SAA_Assay_Combined', 'SAA_Source', 'SAA_Visit',
                 'SAA_Kinetics_Project', 'SAA_Aggregation_Selected',
                 'SAA_Fmax', 'SAA_TTT', 'SAA_AUC', 'SAA_TimeToMaxSlope', 'SAA_MaxSlope',
                 'P237_SAA_Status', 'P237_SAA_Type', 'P237_Event', 'P237_RunDate', 'P237_Replicates_MaxN']
                + [f'P237_{m}_{a}' for m in ['Fmax', 'TTT', 'AUC', 'TimeToMaxSlope', 'MaxSlope']
                   for a in ['mean', 'median', 'n']]
                + ['P299_SAA_Result_Types', 'P299_Event', 'P299_RunDate', 'P299_Replicate_Rows']
                + [f'P299_{m}_{a}' for m in ['Fmax', 'TTT', 'AUC', 'TimeToMaxSlope', 'MaxSlope']
                   for a in ['mean', 'median', 'n']]
                + ['P155_SAA_Status', 'P155_SAA_Type', 'P155_Event', 'P155_RunDate', 'P155_Replicate_Rows']
                + [f'P155_{m}_{a}' for m in ['Fmax', 'TTT', 'AUC', 'TimeToMaxSlope', 'MaxSlope']
                   for a in ['mean', 'median', 'n']])

    core_cols = [c for c in CORE_CURATED_COLUMNS if c in headers] + ['analytic_subgroup_label'] + saa_cols
    all_cols  = headers + ['analytic_subgroup_label'] + saa_cols

    core_rows, all_rows = [], []
    for p in final_ids:
        b = baseline[p].copy()
        b['analytic_subgroup_label'] = GROUP_LABELS.get(b.get('analytic_subgroup', ''), b.get('analytic_subgroup', ''))
        code = b['CSFSAA']
        b['SAA_Status'] = 'Positive' if code in POSITIVE_CODES else 'Negative'
        b['SAA_Status_Code'] = 1 if code in POSITIVE_CODES else 0
        b['CSFSAA_code'] = code
        b['CSFSAA_code_label'] = CODE_LABELS.get(code, '')
        b['SAA_Assay_Combined'] = b.get('CSFSAA_assay', '')
        b['SAA_Source'] = 'Curated CSFSAA @ BL (assay-combined)'
        b['SAA_Visit'] = 'BL'
        b['SAA_Aggregation_Selected'] = REPLICATE_AGGREGATION
        b['BL_enroll_year'] = visit_year(b.get('visit_date', ''))

        r237, r299, r155 = k237.get(p, {}), k299.get(p, {}), k155.get(p, {})
        b.update(r237); b.update(r299); b.update(r155)
        # kinetics preference mirrors the curated assay where available
        if r299:
            pref, proj = 'P299', '299'
        elif r237:
            pref, proj = 'P237', '237'
        elif r155:
            pref, proj = 'P155', '155'
        else:
            pref, proj = None, ''
        b['SAA_Kinetics_Project'] = proj
        for m in ['Fmax', 'TTT', 'AUC', 'TimeToMaxSlope', 'MaxSlope']:
            b[f'SAA_{m}'] = b.get(f'{pref}_{m}_{REPLICATE_AGGREGATION}', '') if pref else ''

        core_rows.append({c: b.get(c, '') for c in core_cols})
        all_rows.append({c: b.get(c, '') for c in all_cols})

    # --- subgroup counts ---
    count_rows = []
    for g in ['A1', 'A2', 'B', 'C', '']:
        sub = [r for r in core_rows if r.get('analytic_subgroup', '') == g]
        pos = sum(r['SAA_Status'] == 'Positive' for r in sub)
        msa = sum(r['CSFSAA_code'] == '3' for r in sub)
        count_rows.append({'analytic_subgroup': g or 'Missing/Unclassified',
                           'description': GROUP_LABELS[g], 'n': len(sub),
                           'Positive': pos, 'Negative': len(sub) - pos,
                           'Positive_pct': round(100 * pos / len(sub), 1) if sub else '',
                           'of_which_MSA_like': msa,
                           'with_kinetics': sum(r['SAA_Fmax'] != '' for r in sub)})
    tot = {k: sum(r[k] for r in count_rows) for k in ('n', 'Positive', 'Negative', 'of_which_MSA_like', 'with_kinetics')}
    tot['Positive_pct'] = round(100 * tot['Positive'] / tot['n'], 1)
    count_rows.append({'analytic_subgroup': 'Total', 'description': 'Baseline-anchored analytic sample', **tot})

    # ---------------------------------------------------------------
    # LONG FILE: one row per participant-visit, baseline SAA carried
    # forward on every row as a fixed grouping variable (*_BL suffix).
    # ---------------------------------------------------------------
    keep = set(final_ids)
    by_pat = {}
    for v in visits:
        if v['PATNO'] in keep:
            by_pat.setdefault(v['PATNO'], []).append(v)

    bl_block = ['SAA_Status_BL', 'SAA_Status_Code_BL', 'CSFSAA_code_BL', 'CSFSAA_code_label_BL', 'BL_enroll_year',
                'SAA_Assay_Combined_BL', 'SAA_Kinetics_Project_BL',
                'SAA_Fmax_BL', 'SAA_TTT_BL', 'SAA_AUC_BL', 'SAA_TimeToMaxSlope_BL', 'SAA_MaxSlope_BL']
    time_block = ['visit_n', 'n_visits', 'is_baseline', 'years_from_BL', 'years_from_BL_exact',
                  'followup_years', 'last_EVENT_ID',
                  'CSFSAA_visit', 'CSFSAA_status_visit', 'seroconverted_by_visit']
    wide_by_pat = {r['PATNO']: r for r in core_rows}

    long_cols = (['PATNO', 'EVENT_ID', 'YEAR'] + time_block + bl_block
                 + [c for c in CORE_CURATED_COLUMNS if c in headers and c not in ('PATNO', 'EVENT_ID', 'YEAR')]
                 + ['analytic_subgroup_label'])

    long_rows = []
    for p in final_ids:
        vs = sorted(by_pat.get(p, []), key=lambda r: VISIT_ORDER.get(r['EVENT_ID'], 999))
        if not vs: continue
        w = wide_by_pat[p]
        bl_age = num(vs[0].get('age_at_visit', ''))
        last_ev = vs[-1]['EVENT_ID']
        fu = num(vs[-1].get('YEAR', ''))
        seen_pos = False
        for i, v in enumerate(vs, start=1):
            r = v.copy()
            r['analytic_subgroup_label'] = GROUP_LABELS.get(r.get('analytic_subgroup', ''), r.get('analytic_subgroup', ''))
            r['visit_n'] = i
            r['n_visits'] = len(vs)
            r['is_baseline'] = 1 if v['EVENT_ID'] == 'BL' else 0
            r['years_from_BL'] = v.get('YEAR', '')
            a = num(v.get('age_at_visit', ''))
            r['years_from_BL_exact'] = round(a - bl_age, 2) if (a is not None and bl_age is not None) else ''
            r['followup_years'] = fu if fu is not None else ''
            r['last_EVENT_ID'] = last_ev

            code_v = v.get('CSFSAA', '')
            r['CSFSAA_visit'] = code_v
            r['CSFSAA_status_visit'] = ('Positive' if code_v in POSITIVE_CODES
                                        else 'Negative' if code_v in NEGATIVE_CODES else '')
            if code_v in POSITIVE_CODES: seen_pos = True
            r['seroconverted_by_visit'] = 1 if (seen_pos and w['SAA_Status'] == 'Negative') else 0

            r['SAA_Status_BL'] = w['SAA_Status']
            r['SAA_Status_Code_BL'] = w['SAA_Status_Code']
            r['CSFSAA_code_BL'] = w['CSFSAA_code']
            r['CSFSAA_code_label_BL'] = w['CSFSAA_code_label']
            r['BL_enroll_year'] = w['BL_enroll_year']
            r['SAA_Assay_Combined_BL'] = w['SAA_Assay_Combined']
            r['SAA_Kinetics_Project_BL'] = w['SAA_Kinetics_Project']
            for m in ['Fmax', 'TTT', 'AUC', 'TimeToMaxSlope', 'MaxSlope']:
                r[f'SAA_{m}_BL'] = w[f'SAA_{m}']
            long_rows.append({c: r.get(c, '') for c in long_cols})

    enroll_tab = []
    for y in sorted({r['BL_enroll_year'] for r in core_rows if r['BL_enroll_year']}):
        sub = [r for r in core_rows if r['BL_enroll_year'] == y]
        pos = sum(r['SAA_Status'] == 'Positive' for r in sub)
        enroll_tab.append({'BL_enroll_year': y, 'n': len(sub), 'Positive': pos,
                           'Negative': len(sub) - pos,
                           'Positive_pct': round(100 * pos / len(sub), 1)})

    visit_tab = []
    for ev, yr in sorted(VISIT_ORDER.items(), key=lambda kv: kv[1]):
        sub = [r for r in long_rows if r['EVENT_ID'] == ev]
        visit_tab.append({'EVENT_ID': ev, 'YEAR': yr, 'n_rows': len(sub),
                          'n_SAA_positive_BL': sum(r['SAA_Status_BL'] == 'Positive' for r in sub),
                          'n_SAA_negative_BL': sum(r['SAA_Status_BL'] == 'Negative' for r in sub)})

    def write_csv(path, cols, rows):
        with path.open('w', encoding='utf-8-sig', newline='') as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction='ignore')
            w.writeheader(); w.writerows(rows)

    write_csv(OUT / 'prodromal_saa_baseline_core.csv', core_cols, core_rows)
    write_csv(OUT / 'prodromal_saa_baseline_all_curated.csv', all_cols, all_rows)
    write_csv(OUT / 'analytic_group_counts_baseline.csv', list(count_rows[0]), count_rows)
    write_csv(OUT / 'prodromal_saa_longitudinal.csv', long_cols, long_rows)
    write_csv(OUT / 'visit_availability.csv',
              ['EVENT_ID', 'YEAR', 'n_rows', 'n_SAA_positive_BL', 'n_SAA_negative_BL'], visit_tab)
    write_csv(OUT / 'positivity_by_enrollment_year.csv',
              ['BL_enroll_year', 'n', 'Positive', 'Negative', 'Positive_pct'], enroll_tab)

    meta = {
        'design': 'baseline-anchored; status from curated CSFSAA on the BL row; positive is type-agnostic (codes 1 and 3)',
        'baseline_events_for_kinetics': sorted(BASELINE_EVENTS),
        'phase2_prodromal_baseline_n': len(all_ids),
        'final_n': len(final_ids),
        'excluded_n': len(all_ids) - len(final_ids),
        'excluded_by_code': dict(excluded),
        'final_positive_n': sum(r['SAA_Status'] == 'Positive' for r in core_rows),
        'final_negative_n': sum(r['SAA_Status'] == 'Negative' for r in core_rows),
        'positive_lbd_like_n': sum(r['CSFSAA_code'] == '1' for r in core_rows),
        'positive_msa_like_n': sum(r['CSFSAA_code'] == '3' for r in core_rows),
        'assay_behind_curated_status': dict(Counter(r['SAA_Assay_Combined'] or '<blank>' for r in core_rows)),
        'with_baseline_kinetics_155_n': len(k155),
        'with_baseline_kinetics_237_n': len(k237),
        'with_baseline_kinetics_299_n': len(k299),
        'with_any_baseline_kinetics_n': sum(r['SAA_Fmax'] != '' for r in core_rows),
        'without_baseline_kinetics_n': sum(r['SAA_Fmax'] == '' for r in core_rows),
        'baseline_covariates_and_status_from_same_visit': True,
        'longitudinal_rows': len(long_rows),
        'longitudinal_participants': len({r['PATNO'] for r in long_rows}),
        'visits_per_participant': dict(sorted(Counter(
            Counter(r['PATNO'] for r in long_rows).values()).items())),
        'rows_per_visit': {r['EVENT_ID']: r['n_rows'] for r in visit_tab},
        'participants_with_followup': sum(1 for p, n in Counter(r['PATNO'] for r in long_rows).items() if n > 1),
        'seroconversion_rows_BLnegative_later_positive': sum(r['seroconverted_by_visit'] for r in long_rows),
        'seroconverted_participants': len({r['PATNO'] for r in long_rows if r['seroconverted_by_visit'] == 1}),
        'positivity_by_enrollment_year': {r['BL_enroll_year']: r['Positive_pct'] for r in enroll_tab},
        'WARNING_ascertainment': ('positivity rises from ~51% (2022) to 100% (2026) enrollees; '
                                  'recent enrollees appear to have positives returned before negatives. '
                                  'Consider restricting to BL_enroll_year <= 2024 for unbiased prevalence.'),
    }
    (OUT / 'metadata_baseline.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    print(json.dumps(meta, indent=2))
    print()
    for r in count_rows:
        print(f"  {r['analytic_subgroup']:<22} n={r['n']:<6} Pos={r['Positive']:<6} Neg={r['Negative']:<6} "
              f"{r['Positive_pct']}%  MSA={r['of_which_MSA_like']}  kinetics={r['with_kinetics']}")
    print()
    for r in visit_tab:
        print(f"  {r['EVENT_ID']:<5} year {r['YEAR']}   rows={r['n_rows']:<6} "
              f"SAA+ {r['n_SAA_positive_BL']:<6} SAA- {r['n_SAA_negative_BL']}")
    print()
    for r in enroll_tab:
        print(f"  enrolled {r['BL_enroll_year']}   n={r['n']:<6} Pos={r['Positive']:<6} "
              f"Neg={r['Negative']:<5} {r['Positive_pct']}%")


build()